# Exploratory Data Analysis — E-Commerce Recommendation System

This notebook performs a structured exploratory data analysis (EDA) on the raw marketplace tables in `../data/raw/` to inform data splitting, candidate retrieval, feature engineering, position-debiased ranking, and serving-time filtering:
1. **Environment Setup & Data Loading** — Load all six Parquet tables (`items`, `users`, `interactions`, `impressions`, `categories`, `sellers`) and inspect shapes.
2. **Schema & Sample Inspection** — Verify column types, primary keys, and sample records across every entity.
3. **Data Quality & Referential Integrity** — Check nulls, exact duplicates, and foreign-key constraints across interaction and catalog tables.
4. **Temporal Coverage & Split Readiness** — Validate interaction/impression time windows and evaluate temporal train/validation/test cutoffs.
5. **Interaction Funnel & Implicit Feedback** — Quantify event-type distributions (`view`, `add_to_cart`, `wishlist`, `purchase`, `rating`) and implicit confidence weights.
6. **Impression Position Bias & Propensity** — Measure slot CTR decay across positions 1–10 and compute Inverse Propensity Weights (IPW).
7. **Sparsity, Long-Tail Concentration & User Activity** — Measure user–item matrix sparsity, Pareto item concentration, and cold-start user rates.
8. **Catalog, Category Hierarchy & Seller Health** — Inspect parent-to-leaf category mappings, price/stock distributions, and visual diagnostics.

## 1. Environment Setup & Data Loading
Load all raw Parquet datasets using Polars and configure table display formatting.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(50)
sns.set_theme(style="whitegrid", palette="deep")

DATA_DIR = Path("../data/raw")

items = pl.read_parquet(DATA_DIR / "items.parquet")
users = pl.read_parquet(DATA_DIR / "users.parquet")
interactions = pl.read_parquet(DATA_DIR / "interactions.parquet")
impressions = pl.read_parquet(DATA_DIR / "impressions.parquet")
categories = pl.read_parquet(DATA_DIR / "categories.parquet")
sellers = pl.read_parquet(DATA_DIR / "sellers.parquet")

datasets: dict[str, pl.DataFrame] = {
    "items": items,
    "users": users,
    "interactions": interactions,
    "impressions": impressions,
    "categories": categories,
    "sellers": sellers,
}

for name, df in datasets.items():
    print(f"{name.capitalize()}: {df.shape}")

Items: (4000, 8)
Users: (8000, 5)
Interactions: (278841, 6)
Impressions: (718850, 6)
Categories: (48, 3)
Sellers: (250, 4)


## 2. Schema & Sample Record Inspection
Inspect the first 5 rows and column data types of each table to verify primary keys, timestamp precision, and categorical attributes.

In [2]:
# Catalog items: price, brand, category, seller, and current stock
items.head(5)

item_id,title,category_id,brand,price,seller_id,created_at,stock
i64,str,i64,str,f64,i64,datetime[ns],i64
1,"""HexaPlay Prime Building Sets 1""",40,"""HexaPlay""",90.33,135,2025-06-27 00:00:00,42
2,"""BrickWorks Classic Outdoor Pla…",43,"""BrickWorks""",57.08,146,2025-10-31 00:00:00,83
3,"""PixelCraft Compact Smartphones…",10,"""PixelCraft""",388.25,218,2025-05-08 00:00:00,104
4,"""BladeCraft Signature Grooming …",32,"""BladeCraft""",42.93,134,2025-05-15 00:00:00,37
5,"""VelvetHue Ultra Grooming Kits …",32,"""VelvetHue""",29.2,126,2025-12-20 00:00:00,6


In [3]:
# User profiles: signup timestamp, city, device, and preferred language
users.head(5)

user_id,signup_date,city,device,language
i64,datetime[ns],str,str,str
1,2025-10-13 00:00:00,"""Austin""","""ios""","""en"""
2,2024-08-18 00:00:00,"""Toronto""","""web""","""en"""
3,2025-10-08 00:00:00,"""Austin""","""web""","""de"""
4,2025-01-29 00:00:00,"""Cairo""","""android""","""es"""
5,2026-03-19 00:00:00,"""Madrid""","""android""","""en"""


In [4]:
# User-item interactions: explicit ratings and implicit funnel events
interactions.head(5)

user_id,item_id,event_type,timestamp,session_id,rating
i64,i64,str,datetime[ns],str,f64
3,3591,"""view""",2026-01-01 20:14:02,"""s_1""",null
3,3993,"""view""",2026-01-01 20:14:50,"""s_1""",null
3,3993,"""add_to_cart""",2026-01-01 20:15:15,"""s_1""",null
3,3993,"""purchase""",2026-01-01 20:15:45,"""s_1""",null
12,2222,"""view""",2026-01-01 15:13:21,"""s_2""",null


In [5]:
# Exposure impressions: slate position (1-10) and binary click indicator
impressions.head(5)

user_id,item_id,session_id,position,clicked,timestamp
i64,i64,str,i64,i64,datetime[ns]
3,3229,"""s_1""",1,0,2026-01-01 20:13:26
3,265,"""s_1""",2,0,2026-01-01 20:13:27
3,3591,"""s_1""",3,1,2026-01-01 20:13:28
3,3451,"""s_1""",4,0,2026-01-01 20:13:29
3,3705,"""s_1""",5,0,2026-01-01 20:13:30


In [6]:
# Category taxonomy: two-level hierarchy (parent_id == null indicates root category)
categories.head(5)

category_id,name,parent_id
i64,str,i64
1,"""Electronics""",null
9,"""Headphones & Audio""",1
10,"""Smartphones & Accessories""",1
11,"""Laptops & Tablets""",1
12,"""Smart Home & Cameras""",1


In [7]:
# Marketplace sellers: aggregate rating and onboarding date
sellers.head(5)

seller_id,name,rating,joined_at
i64,str,f64,datetime[ns]
1,"""Apex Direct #1""",4.76,2023-12-25 00:00:00
2,"""Nova Direct #2""",4.82,2023-06-18 00:00:00
3,"""Blue Direct #3""",4.61,2023-12-05 00:00:00
4,"""Prime Direct #4""",4.33,2024-08-10 00:00:00
5,"""Atlas Direct #5""",4.52,2023-11-05 00:00:00


## 3. Data Quality & Referential Integrity Audit
Verify that primary tables contain no unexpected nulls or duplicate rows, confirm that `interactions.rating` is only populated on `'rating'` events, and validate foreign-key integrity across all relational joins.

In [8]:
for name, df in [
    ("items", items),
    ("users", users),
    ("interactions", interactions),
    ("impressions", impressions),
    ("categories", categories),
    ("sellers", sellers),
]:
    print(f"{name} nulls: {df.null_count().row(0)} | dups: {df.is_duplicated().sum()}")

items nulls: (0, 0, 0, 0, 0, 0, 0, 0) | dups: 0
users nulls: (0, 0, 0, 0, 0) | dups: 0
interactions nulls: (0, 0, 0, 0, 0, 265924) | dups: 0
impressions nulls: (0, 0, 0, 0, 0, 0) | dups: 0
sellers nulls: (0, 0, 0, 0) | dups: 0


In [9]:
referential_checks = {
    "bad user_id (inter)": interactions.filter(~pl.col("user_id").is_in(users["user_id"])).height,
    "bad item_id (inter)": interactions.filter(~pl.col("item_id").is_in(items["item_id"])).height,
    "bad user_id (impr)": impressions.filter(~pl.col("user_id").is_in(users["user_id"])).height,
    "bad item_id (impr)": impressions.filter(~pl.col("item_id").is_in(items["item_id"])).height,
    "bad category": items.filter(~pl.col("category_id").is_in(categories["category_id"])).height,
    "bad seller": items.filter(~pl.col("seller_id").is_in(sellers["seller_id"])).height,
}
for check_name, count in referential_checks.items():
    print(f"{check_name}: {count}")

bad user_id (inter): 0
bad item_id (inter): 0
bad item_id (impr): 0
bad category: 0
bad seller: 0


## 4. Temporal Coverage, Cutoff Windows & Leakage Checks
Both `interactions` and `impressions` span the 90-day Q1 2026 window (`2026-01-01` to `2026-04-01`). We inspect the exact timestamp bounds and compute temporal split cutoffs (e.g., 70% Train / 15% Validation / 15% Test) along with cold-start user/item rates in downstream evaluation windows.

In [10]:
print("Interactions window:", interactions["timestamp"].min(), "->", interactions["timestamp"].max())
print("Impressions window: ", impressions["timestamp"].min(), "->", impressions["timestamp"].max())

2026-01-01 00:07:44 2026-04-01 00:48:59
2026-01-01 00:07:24 2026-03-31 23:59:28


In [ ]:
# Evaluate a 70% / 15% / 15% quantile timestamp split for leakage-free evaluation
sorted_ts = interactions.sort("timestamp")["timestamp"]
train_cutoff = sorted_ts[int(len(sorted_ts) * 0.70)]
val_cutoff = sorted_ts[int(len(sorted_ts) * 0.85)]

train_df = interactions.filter(pl.col("timestamp") <= train_cutoff)
val_df = interactions.filter((pl.col("timestamp") > train_cutoff) & (pl.col("timestamp") <= val_cutoff))
test_df = interactions.filter(pl.col("timestamp") > val_cutoff)

train_users = set(train_df["user_id"].unique())
train_items = set(train_df["item_id"].unique())

split_summary = pl.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [train_df.height, val_df.height, test_df.height],
    "start_ts": [train_df["timestamp"].min(), val_df["timestamp"].min(), test_df["timestamp"].min()],
    "end_ts": [train_df["timestamp"].max(), val_df["timestamp"].max(), test_df["timestamp"].max()],
    "unique_users": [train_df["user_id"].n_unique(), val_df["user_id"].n_unique(), test_df["user_id"].n_unique()],
    "unseen_users_vs_train": [
        0,
        val_df.filter(~pl.col("user_id").is_in(list(train_users)))["user_id"].n_unique(),
        test_df.filter(~pl.col("user_id").is_in(list(train_users)))["user_id"].n_unique(),
    ],
    "unseen_items_vs_train": [
        0,
        val_df.filter(~pl.col("item_id").is_in(list(train_items)))["item_id"].n_unique(),
        test_df.filter(~pl.col("item_id").is_in(list(train_items)))["item_id"].n_unique(),
    ],
})
split_summary

## 5. Interaction Funnel & Implicit Feedback Weighting
Explicit `rating` events make up only **4.63%** (12,917 / 278,841) of interactions. For candidate retrieval (Matrix Factorization / Two-Tower), combining all interaction types with confidence weights (`view = 1.0`, `wishlist = 2.0`, `add_to_cart = 3.0`, `purchase = 5.0`, `rating >= 4 = 4.0`) captures the full conversion funnel.

In [11]:
event_funnel = (
    interactions.group_by("event_type")
    .len()
    .sort("len", descending=True)
    .with_columns(
        (pl.col("len") / interactions.height * 100).round(2).alias("share_pct")
    )
)
event_funnel

event_type,len
str,u32
"""view""",127542
"""add_to_cart""",56218
"""wishlist""",45443
"""purchase""",36721
"""rating""",12917


In [ ]:
# Inspect explicit rating distribution on the 12,917 rating events
rating_events = interactions.filter(pl.col("event_type") == "rating")
print(rating_events["rating"].describe())

# Session-level funnel conversion rates
session_funnel = (
    interactions.group_by("session_id")
    .agg([
        (pl.col("event_type") == "view").any().alias("has_view"),
        (pl.col("event_type") == "add_to_cart").any().alias("has_cart"),
        (pl.col("event_type") == "purchase").any().alias("has_purchase"),
    ])
    .select([
        pl.len().alias("total_sessions"),
        pl.col("has_view").mean().round(4).alias("session_view_rate"),
        pl.col("has_cart").mean().round(4).alias("session_cart_rate"),
        pl.col("has_purchase").mean().round(4).alias("session_purchase_rate"),
    ])
)
session_funnel

## 6. Impression Position Bias & Propensity Analysis
Click-through rate (CTR) drops monotonically from **34.26%** at Position 1 to **4.14%** at Position 10 (an **8.27×** decay). We compute empirical examination propensities $p_k = \text{CTR}_k / \text{CTR}_1$ and stabilized Inverse Propensity Weights (IPW) for debiased ranker training.

In [12]:
position_ctr = (
    impressions.group_by("position")
    .agg([
        pl.len().alias("impressions"),
        pl.col("clicked").sum().alias("clicks"),
        pl.col("clicked").mean().alias("ctr"),
    ])
    .sort("position")
    .with_columns([
        (pl.col("ctr") / pl.col("ctr").first()).round(4).alias("rel_propensity"),
        (pl.col("ctr").first() / pl.col("ctr")).clip(1.0, 10.0).round(4).alias("ipw_weight"),
    ])
)
position_ctr.select(["position", "ctr"])

position,ctr
i64,f64
1,0.342645
2,0.232079
3,0.177895
4,0.142617
5,0.115824
6,0.097531
7,0.080754
8,0.066857
9,0.055491


## 7. Sparsity, Long-Tail Item Concentration & User Activity
Evaluate user–item interaction density, Pareto concentration across the 4,000-item catalog, and user activity quantiles (including cold-start user counts).

In [13]:
item_popularity = interactions.group_by("item_id").len().sort("len", descending=True)
cum_share = item_popularity["len"].cum_sum() / item_popularity["len"].sum()
items_for_80pct = (cum_share < 0.80).sum() / items.height
print(f"items for 80%: {items_for_80pct:.3f}")

items for 80%: 0.469


In [14]:
n_pairs = interactions.select(["user_id", "item_id"]).unique().height
sparsity = 1.0 - n_pairs / (users.height * items.height)
print(f"sparsity: {sparsity}")

sparsity: 0.996162875


In [15]:
user_activity = interactions.group_by("user_id").len()
cold_start_users = users.height - user_activity.height
print(f"Active users: {user_activity.height} / {users.height} | Cold-start users: {cold_start_users} ({cold_start_users / users.height:.2%})")
user_activity["len"].describe()

statistic,value
str,f64
"""count""",7506.0
"""null_count""",0.0
"""mean""",37.149081
"""std""",32.991304
"""min""",1.0
"""25%""",15.0
"""50%""",28.0
"""75%""",49.0
"""max""",297.0


## 8. Catalog Health & Visual Diagnostics
Inspect parent vs. leaf category mappings, out-of-stock item counts (`stock == 0`), price distributions, and plot key diagnostic curves (Position CTR decay, Item Pareto curve, and User Activity distribution).

In [ ]:
# Parent vs. leaf categories and stock availability
root_cats = categories.filter(pl.col("parent_id").is_null())
leaf_cats = categories.filter(pl.col("parent_id").is_not_null())
out_of_stock = items.filter(pl.col("stock") <= 0).height

print(f"Root categories: {root_cats.height} | Leaf categories: {leaf_cats.height}")
print(f"Out-of-stock items (stock <= 0): {out_of_stock} / {items.height} ({out_of_stock / items.height:.2%})")
print(items.select(["price", "stock"]).describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Position Bias CTR Decay
pos_df = position_ctr.to_pandas()
axes[0].plot(pos_df["position"], pos_df["ctr"] * 100, marker="o", linewidth=2, color="#1f77b4")
axes[0].set_title("Impression CTR by Slate Position")
axes[0].set_xlabel("Position (1 = Top Slot)")
axes[0].set_ylabel("Click-Through Rate (%)")
axes[0].set_xticks(range(1, 11))

# 2. Cumulative Item Interaction Share (Pareto Curve)
item_fraction = np.arange(1, len(cum_share) + 1) / items.height * 100
axes[1].plot(item_fraction, cum_share.to_numpy() * 100, linewidth=2, color="#2ca02c")
axes[1].axhline(80, color="crimson", linestyle="--", alpha=0.7, label="80% Interactions")
axes[1].axvline(items_for_80pct * 100, color="gray", linestyle="--", alpha=0.7, label=f"{items_for_80pct*100:.1f}% Items")
axes[1].set_title("Cumulative Interactions vs. Catalog %")
axes[1].set_xlabel("Percentage of Catalog Items (%)")
axes[1].set_ylabel("Cumulative Share of Interactions (%)")
axes[1].legend()

# 3. User Interaction Count Distribution
axes[2].hist(user_activity["len"].to_numpy(), bins=40, color="#ff7f0e", edgecolor="white")
axes[2].axvline(user_activity["len"].median(), color="black", linestyle="--", label=f"Median = {user_activity['len'].median():.0f}")
axes[2].set_title("User Activity Distribution (Active Users)")
axes[2].set_xlabel("Interactions per User")
axes[2].set_ylabel("Number of Users")
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. Key Takeaways & Production Pipeline Decisions

| Component | Empirical Finding | Actionable Design Decision |
| :--- | :--- | :--- |
| **Data Splitting** | 90-day window (`2026-01-01` to `2026-04-01`); zero orphan keys or duplicate rows | Use strict chronological cutoffs (`train` / `val` / `test`) rather than random splits to prevent future leakage. |
| **Cold-Start Handling** | 494 / 8,000 users (6.18%) have zero interactions | Build a popularity + demographic segment fallback (`city`, `device`, `language`) for unseen users. |
| **Candidate Retrieval** | 99.62% sparsity; explicit ratings represent only 4.63% of events | Train retrieval on weighted implicit feedback (`purchase=5`, `add_to_cart=3`, `wishlist=2`, `view=1`). |
| **Ranking Debiasing** | Position 1 CTR (34.26%) is 8.27× Position 10 CTR (4.14%) | Feed `position` as a dedicated bias feature during training and clamp `position = 1` at serving time. |
| **Post-Ranking & Serving** | 46.9% of items account for 80% of interactions; items carry real-time `stock` | Filter `stock <= 0` before ranking and apply category/seller diversity reranking on the top-$K$ slate. |